In [1]:
import os, pickle
import pandas as pd

In [2]:
import sys
sys.path.append("../../training_data")

In [3]:
from utils.utils import Cif

# Data

In [4]:
with open("../../training_data/7.Extra_set/features.pkl", "rb") as f:
    news = pickle.load(f)

len(news), news

(35,
 {'22mj':     Residues                                                          \
           pdb label_entity_id label_asym_id label_seq_id auth_asym_id   
  0       22mj               1             A           10            A   
  1       22mj               1             A           11            A   
  2       22mj               1             A           12            A   
  3       22mj               1             A           13            A   
  4       22mj               1             A           14            A   
  ..       ...             ...           ...          ...          ...   
  281     22mj               1             A          315            A   
  282     22mj               1             A          316            A   
  283     22mj               1             A          317            A   
  284     22mj               1             A          318            A   
  285     22mj               1             A          319            A   
  
                      

In [5]:
with open("../../training_data/7.Extra_set/news_sites.pkl", "rb") as f:
    news_sites = {k: v for k, v in pickle.load(f).items() if k in news}

len(news_sites), news_sites

(35,
 {'22mj': [{'mod':      label_comp_id label_asym_id label_entity_id label_seq_id  \
    2394         A1MDP             H               4            .   
    
         pdbx_PDB_ins_code auth_seq_id auth_comp_id auth_asym_id  \
    2394                 ?         407        A1MDP            A   
    
         pdbx_PDB_model_num pdbx_label_index  
    2394                  1              407  ,
    'site':    label_comp_id label_asym_id label_entity_id label_seq_id pdbx_PDB_ins_code  \
    0            THR             A               1          118                 ?   
    1            PRO             A               1          119                 ?   
    2            THR             A               1          120                 ?   
    3            LYS             A               1          129                 ?   
    4            GLY             A               1          130                 ?   
    5            GLY             A               1          131                 ?  

# Process structures

In [6]:
os.makedirs("structures", exist_ok=True)

In [7]:
import pymol2

In [8]:
for pdb in news:
    ciff = f"../../training_data/7.Extra_set/cifs/{pdb}.cif"
    if not os.path.isfile(f"structures/{pdb}.cif"):
        os.system(f"cp {ciff} structures/{pdb}.cif")

    if not os.path.isfile(f"structures/{pdb}.pdb"):
        with pymol2.PyMOL() as pymol:
            pymol.cmd.feedback("disable", "executive", "details")
            pymol.cmd.load(ciff, "structure")
            pymol.cmd.save(f"structures/{pdb}.pdb", "structure")

# Our results

In [9]:
from autogluon.tabular import TabularDataset, TabularPredictor

In [10]:
def process_dataframe(df):
    df.index = df["Pockets"][["pdb", "pocket"]].apply(lambda x: "_".join(x), axis=1)
    df = df.drop(columns=["Pockets"], level=0)
    df.columns = map(lambda x: "_".join(x), df.columns.values)
    df.loc[:,'Label_label'] = df['Label_label'].astype("category")
    return df

## Model 5.

In [11]:
predictor = TabularPredictor.load("../pockets_physchem_deploy")

In [12]:
with open("../../training_data/7.Extra_set/pockets_features.pkl", "rb") as f:
    extra = pd.concat((
        pickle.load(f).values()
    ))

extra

Pockets                                              Label      FPocket  \
        pdb    pocket nres site_in_pocket pocket_in_site label Pocket Score   
0      22mj  pocket11   13       0.000000       0.000000     0      -0.0344   
1      22mj   pocket7   10       0.000000       0.000000     0       0.0487   
2      22mj   pocket1   17       0.000000       0.000000     0       0.2733   
3      22mj   pocket2   44       0.928571       0.590909     1       0.2570   
4      22mj   pocket4   10       0.000000       0.000000     0       0.0698   
..      ...       ...  ...            ...            ...   ...          ...   
710    9ow9   pocket3   16       0.055556       0.125000     0       0.1895   
711    9ow9  pocket12   14       0.000000       0.000000     0       0.0593   
712    9ow9  pocket19   13       0.111111       0.307692     0      -0.0406   
713    9ow9  pocket21   13       0.000000       0.000000     0      -0.1540   
714    9ow9   pocket6   14       0.083333       0.214286     0       0.1175   

                                                                 ...  \
    Drug Score Number of alpha spheres Mean alpha-sphere radius  ...   
0       0.0005                    40.0                   3.6645  ...   
1       0.0007                    40.0                   3.5474  ...   
2       0.1102                    49.0                   3.5212  ...   
3       0.9974                   312.0                   3.5983  ...   
4       0.0009                    59.0                   3.5695  ...   
..         ...                     ...                      ...  ...   
710     0.0215                    64.0                   3.4302  ...   
711     0.1223                    57.0                   3.3835  ...   
712     0.1032                    51.0                   3.5753  ...   
713     0.0179                    61.0                   3.6792  ...   
714     0.0244                    59.0                   3.3881  ...   

      HHBlits                                                              \
         M->M      M->I      M->D      I->M      I->I      D->M      D->D   
0    0.965538  0.017878  0.016567  0.293938  0.706008  0.220269  0.779680   
1    0.976615  0.014044  0.009283  0.198701  0.801308  0.253872  0.746113   
2    0.972468  0.022047  0.005504  0.250170  0.573332  0.291624  0.708348   
3    0.955549  0.013536  0.030923  0.319427  0.680553  0.187262  0.812706   
4    0.935202  0.008965  0.055771  0.259235  0.740732  0.055374  0.944506   
..        ...       ...       ...       ...       ...       ...       ...   
710  0.979323  0.015552  0.005174  0.221978  0.715481  0.150538  0.849464   
711  0.979097  0.012785  0.008063  0.281685  0.718355  0.214290  0.785698   
712  0.986117  0.006892  0.006934  0.240089  0.759958  0.193603  0.806453   
713  0.983233  0.009535  0.007285  0.346869  0.653202  0.191585  0.808504   
714  0.965064  0.026202  0.008636  0.254808  0.745237  0.221290  0.778718   

                                    
          Neff    Neff_I    Neff_D  
0    13.560923  1.422769  2.255692  
1    13.655300  1.368300  3.001700  
2    13.637059  1.369588  2.838706  
3    13.756750  1.343114  4.087545  
4    13.609500  1.177300  7.705000  
..         ...       ...       ...  
710  12.131875  1.193875  1.410125  
711  12.596214  1.280786  1.704857  
712  12.717846  1.153692  1.917231  
713  11.523231  1.086769  1.785231  
714  12.702857  1.569286  1.871857  

[706 rows x 194 columns]

In [13]:
# Check that all PDBs only have one positive pocket
extra.assign(totalpos=lambda df: df.groupby(("Pockets", "pdb"))[("Label", "label"),].transform("sum")).iloc[:, [0, -1]].drop_duplicates().sort_values("totalpos")

,Pockets,totalpos
,pdb,
31,6on4,0
82,7agj,0
177,7ry1,0
627,9lhf,0
18,6vvq,1
49,6z1m,1
63,6z74,1
12,6s3a,1
72,7ac8,1


In [14]:
extra_probs = predictor.predict_proba( process_dataframe(extra) )
extra_probs

,0,1
22mj_pocket11,0.984074,0.015926
22mj_pocket7,0.999728,0.000272
22mj_pocket1,0.999994,0.000006
22mj_pocket2,0.195944,0.804056
22mj_pocket4,0.999977,0.000023
...,...,...
9ow9_pocket3,0.999598,0.000402
9ow9_pocket12,0.999447,0.000553
9ow9_pocket19,0.999827,0.000173
9ow9_pocket21,0.999992,0.000008


In [15]:
model5_results = {
    pdb: {
        pocket: {
            "prob": prob,
            "pred": 1 if prob >= 0.5 else 0,
            "label": info[("Label", "label")],
            "max_overlap": overlaps.max(),
            **overlaps["Pockets"].to_dict()
        }
        for pocket in pockets["pocket"]
        for prob in (extra_probs.loc[f"{pdb}_{pocket}", 1],)
        for info in (extra.loc[f"{pdb}_{pocket}"],)
        for overlaps in (info[[("Pockets", "pocket_in_site"), ("Pockets", "site_in_pocket")]],)
    }
    for pdb, pockets in (
        pd.DataFrame(
            extra_probs.index.map(lambda x: x.split("_")).values.tolist(),
            columns=["pdb", "pocket"]
        )
        .groupby("pdb")
    )
}

model5_results

{'22mj': {'pocket11': {'prob': 0.015926241874694824,
   'pred': 0,
   'label': 0,
   'max_overlap': 0.0,
   'pocket_in_site': 0.0,
   'site_in_pocket': 0.0},
  'pocket7': {'prob': 0.0002722057106439024,
   'pred': 0,
   'label': 0,
   'max_overlap': 0.0,
   'pocket_in_site': 0.0,
   'site_in_pocket': 0.0},
  'pocket1': {'prob': 5.723202775698155e-06,
   'pred': 0,
   'label': 0,
   'max_overlap': 0.0,
   'pocket_in_site': 0.0,
   'site_in_pocket': 0.0},
  'pocket2': {'prob': 0.80405592918396,
   'pred': 1,
   'label': 1,
   'max_overlap': 0.9285714285714286,
   'pocket_in_site': 0.5909090909090909,
   'site_in_pocket': 0.9285714285714286},
  'pocket4': {'prob': 2.3296684958040714e-05,
   'pred': 0,
   'label': 0,
   'max_overlap': 0.0,
   'pocket_in_site': 0.0,
   'site_in_pocket': 0.0},
  'pocket9': {'prob': 0.000627053901553154,
   'pred': 0,
   'label': 0,
   'max_overlap': 0.08333333333333333,
   'pocket_in_site': 0.08333333333333333,
   'site_in_pocket': 0.03571428571428571},
  'p

In [16]:
pd.DataFrame((
    {
        "pdb": pdb,
        "pocket": pocket,
        **pocketd
    }
    for pdb, pockets in model5_results.items()
    for pocket, pocketd in pockets.items()
)).sort_values(["pred", "label", "prob", "site_in_pocket"], ascending=False).iloc[:40]

,pdb,pocket,prob,pred,label,max_overlap,pocket_in_site,site_in_pocket
657,9ntc,pocket1,0.982443,1,1,0.717391,0.717391,0.647059
532,8v81,pocket1,0.977954,1,1,0.962963,0.208000,0.962963
673,9oul,pocket1,0.950287,1,1,0.903226,0.259259,0.903226
498,8uk6,pocket1,0.946666,1,1,0.952381,0.540541,0.952381
30,6vvq,pocket1,0.929778,1,1,0.777778,0.777778,0.736842
326,8f4s,pocket1,0.923800,1,1,0.846154,0.733333,0.846154
475,8qni,pocket1,0.917425,1,1,0.966667,0.322222,0.966667
21,6s3a,pocket1,0.916088,1,1,0.740741,0.512821,0.740741
449,8jp0,pocket1,0.906662,1,1,0.781250,0.320513,0.781250
146,7pt0,pocket2,0.900742,1,1,0.863636,0.527778,0.863636


# Results

## AllositePro

In [18]:
allositepro_resultsf = "AllositePro/allositepro_results.pkl"

with open(allositepro_resultsf, "rb") as f:
    allositepro_results = {k: v for k,v in pickle.load(f).items() if k in news}

len(allositepro_results), allositepro_results

(15,
 {'22mj': {'pocket0': {'Volume': 992.587,
    'SASA': 508.691,
    'Druggability Score': 0.868,
    'logitProb': 0.822,
    'nmaScore': 0.196,
    'hitScore': 0.697,
    'residues':    auth_asym_id auth_seq_id
    0             A         164
    1             A         163
    2             A         150
    3             A         146
    4             A         195
    5             A         193
    6             A         118
    7             A         133
    8             A         223
    9             A         153
    10            A         151
    11            A         131
    12            A         272
    13            A         160
    14            A         148
    15            A         167
    16            A         236
    17            A         116
    18            A         189
    19            A         119
    20            A         120
    21            A         222
    22            A         233
    23            A         132
    24           

All output pockets are predicted positive.

In [19]:
allositepro_results = {
    pdb: {
        pocket: {
            "pred": 1,
            **pocketd
        }
        for pocket, pocketd in pockets.items()
    }
    for pdb, pockets in allositepro_results.items()
}
allositepro_results

{'22mj': {'pocket0': {'pred': 1,
   'Volume': 992.587,
   'SASA': 508.691,
   'Druggability Score': 0.868,
   'logitProb': 0.822,
   'nmaScore': 0.196,
   'hitScore': 0.697,
   'residues':    auth_asym_id auth_seq_id
   0             A         164
   1             A         163
   2             A         150
   3             A         146
   4             A         195
   5             A         193
   6             A         118
   7             A         133
   8             A         223
   9             A         153
   10            A         151
   11            A         131
   12            A         272
   13            A         160
   14            A         148
   15            A         167
   16            A         236
   17            A         116
   18            A         189
   19            A         119
   20            A         120
   21            A         222
   22            A         233
   23            A         132
   24            A         156
   25   

## PASSer

In [19]:
passer_resultsf = "PASSer/passer_results.pkl"

with open(passer_resultsf, "rb") as f:
    passer_results = {k1: {k2: v2 for k2, v2 in v1.items() if k2 in news} for k1, v1 in pickle.load(f).items()}

len(passer_results), tuple(len(v) for v in passer_results.values()), passer_results

(3,
 (9, 9, 9),
 {'ensemble': {'7gqu': {'20': {'prob/score': 50.99665205925703,
     'residues':    auth_asym_id auth_seq_id pdbx_PDB_ins_code
     0             A         916                 ?
     1             A         917                 ?
     4             A         920                 ?
     5             A         727                 ?
     6             A         919                 ?
     8             A         555                 ?
     9             A         552                 ?
     11            A         551                 ?
     12            A         846                 ?
     13            A         845                 ?
     15            A         849                 ?
     17            A         913                 ?
     25            A         706                 ?
     26            A         570                 ?
     27            A         726                 ?
     29            A         725                 ?
     30            A         898         

Top 3 pockets for each PDB are predicted positive.

In [20]:
passer_results = {
    model: {
        pdb: {
            pocket: {
                "pred": 1 if i <=2 else 0,
                **pocketd
            }
            for i, (pocket, pocketd) in enumerate( 
                sorted(
                    pockets.items(), 
                    key=lambda x: x[-1]["prob/score"], 
                    reverse=True
                )
            )
        }
        for pdb, pockets in pdbs.items()
    }
    for model, pdbs in passer_results.items()
}
passer_results

{'ensemble': {'7gqu': {'20': {'pred': 1,
    'prob/score': 50.99665205925703,
    'residues':    auth_asym_id auth_seq_id pdbx_PDB_ins_code
    0             A         916                 ?
    1             A         917                 ?
    4             A         920                 ?
    5             A         727                 ?
    6             A         919                 ?
    8             A         555                 ?
    9             A         552                 ?
    11            A         551                 ?
    12            A         846                 ?
    13            A         845                 ?
    15            A         849                 ?
    17            A         913                 ?
    25            A         706                 ?
    26            A         570                 ?
    27            A         726                 ?
    29            A         725                 ?
    30            A         898                 ?
    31    

## DeepAllo

In [21]:
deepallo_resultsf = "DeepAllo/deepallo_results.pkl"

with open(deepallo_resultsf, "rb") as f:
    deepallo_results = pickle.load(f)

len(deepallo_results), deepallo_results

(9,
 {'7gqu': {'pocket1': {'pred': 1,
    'residues':    auth_asym_id auth_seq_id pdbx_PDB_ins_code
    0             A         601                 ?
    1             A         829                 ?
    2             A         573                 ?
    3             A         577                 ?
    4             A         578                 ?
    ..          ...         ...               ...
    73            A         923                 ?
    74            A         551                 ?
    75            A         547                 ?
    76            A         883                 ?
    77            A         801                 ?
    
    [78 rows x 3 columns],
    'prob': 0.2415745109319687},
   'pocket2': {'pred': 1,
    'residues':    auth_asym_id auth_seq_id pdbx_PDB_ins_code
    0             A         916                 ?
    1             A         917                 ?
    2             A         920                 ?
    3             A         727                

In [22]:
deepallo_results["8aq6"]

{'pocket3_G': {'pred': 1,
  'residues':    auth_asym_id auth_seq_id pdbx_PDB_ins_code
  0             G          31                 ?
  1             G          19                 ?
  2             G          18                 ?
  3             G          38                 ?
  4             G          36                 ?
  5             G          29                 ?
  6             G          37                 ?
  7             G          92                 ?
  8             G         162                 ?
  9             G         110                 ?
  10            G          30                 ?
  11            G          22                 ?
  12            G          58                 ?
  13            G          12                 ?,
  'prob': 0.3230498433113098},
 'pocket1_H': {'pred': 1,
  'residues':    auth_asym_id auth_seq_id pdbx_PDB_ins_code
  0             H         110                 ?
  1             H          92                 ?
  2             H          3

In [23]:
deepallo_results = {
    pdb: {
        pocket: {
            "prob": 0, # will be replaced by the real prob of the top3 pockets if it's in pocketd
            **pocketd
        }
        for pocket, pocketd in pockets.items()
    }
    for pdb, pockets in deepallo_results.items()
}
deepallo_results

{'7gqu': {'pocket1': {'prob': 0.2415745109319687,
   'pred': 1,
   'residues':    auth_asym_id auth_seq_id pdbx_PDB_ins_code
   0             A         601                 ?
   1             A         829                 ?
   2             A         573                 ?
   3             A         577                 ?
   4             A         578                 ?
   ..          ...         ...               ...
   73            A         923                 ?
   74            A         551                 ?
   75            A         547                 ?
   76            A         883                 ?
   77            A         801                 ?
   
   [78 rows x 3 columns]},
  'pocket2': {'prob': 0.1223275363445282,
   'pred': 1,
   'residues':    auth_asym_id auth_seq_id pdbx_PDB_ins_code
   0             A         916                 ?
   1             A         917                 ?
   2             A         920                 ?
   3             A         727           

## STINGAllo

In [24]:
stingallo_resultsf = "STINGAllo/stingallo_results.pkl"

with open(stingallo_resultsf, "rb") as f:
    stingallo_results = pickle.load(f)

len(stingallo_results), stingallo_results

(3,
 {'8qni': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         148}},
  '7gqu': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         574
    1            A         576}},
  '8aq6': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            G          91}}})

In [25]:
stingallo_results = {
    pdb: {
        pocket: {
            "pred": 1,
            **pocketd
        }
        for pocket, pocketd in pockets.items()
    }
    for pdb, pockets in stingallo_results.items()
}
stingallo_results

{'8qni': {'pocket': {'pred': 1,
   'residues':   auth_asym_id auth_seq_id
   0            A         148}},
 '7gqu': {'pocket': {'pred': 1,
   'residues':   auth_asym_id auth_seq_id
   0            A         574
   1            A         576}},
 '8aq6': {'pocket': {'pred': 1,
   'residues':   auth_asym_id auth_seq_id
   0            G          91}}}

## AlloFusion

In [26]:
allofusion_resultsf = "AlloFusion/allofusion_results.pkl"

with open(allofusion_resultsf, "rb") as f:
    allofusion_results = pickle.load(f)

len(allofusion_results), allofusion_results

(9,
 {'7gqu': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         577
    1            A         581
    2            A         713
    3            A         721
    4            A         730
    5            A         731
    6            A         854
    7            A         888
    8            A         891}},
  '7yg5': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         381
    1            A        1354
    2            A        1397}},
  '8f4s': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A        6878
    1            A        6880
    2            A        6968
    3            A        7000}},
  '8jp0': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         486
    1            A         809}},
  '8qni': {'pocket': {'residues':   auth_asym_id auth_seq_id
    0            A         132
    1            A         141
    2            A         143
    3            A         2

In [27]:
allofusion_results = {
    pdb: {
        f"pocket_{res['auth_asym_id']}_{res['auth_seq_id']}": {
            "pred": 1,
            "residues": res.to_frame().T
        }
        for pocket, pocketd in pockets.items()
        for i, res in pocketd["residues"].iterrows()
    }
    for pdb, pockets in allofusion_results.items()
}
allofusion_results

{'7gqu': {'pocket_A_577': {'pred': 1,
   'residues':   auth_asym_id auth_seq_id
   0            A         577},
  'pocket_A_581': {'pred': 1,
   'residues':   auth_asym_id auth_seq_id
   1            A         581},
  'pocket_A_713': {'pred': 1,
   'residues':   auth_asym_id auth_seq_id
   2            A         713},
  'pocket_A_721': {'pred': 1,
   'residues':   auth_asym_id auth_seq_id
   3            A         721},
  'pocket_A_730': {'pred': 1,
   'residues':   auth_asym_id auth_seq_id
   4            A         730},
  'pocket_A_731': {'pred': 1,
   'residues':   auth_asym_id auth_seq_id
   5            A         731},
  'pocket_A_854': {'pred': 1,
   'residues':   auth_asym_id auth_seq_id
   6            A         854},
  'pocket_A_888': {'pred': 1,
   'residues':   auth_asym_id auth_seq_id
   7            A         888},
  'pocket_A_891': {'pred': 1,
   'residues':   auth_asym_id auth_seq_id
   8            A         891}},
 '7yg5': {'pocket_A_381': {'pred': 1,
   'residues':   

## AllosES

In [28]:
alloses_resultsf = "AllosES/alloses_results.pkl"

with open(alloses_resultsf, "rb") as f:
    alloses_results = pickle.load(f)

len(alloses_results), alloses_results

(9,
 {'7gqu': {'pocket7': {'pro_ave': 0.6077189166132947,
    'residues':    auth_seq_id auth_asym_id
    0          601            A
    1          829            A
    2          573            A
    3          577            A
    4          578            A
    ..         ...          ...
    73         923            A
    74         551            A
    75         547            A
    76         883            A
    77         801            A
    
    [78 rows x 2 columns]},
   'pocket20': {'pro_ave': 0.4671694797381203,
    'residues':    auth_seq_id auth_asym_id
    0          916            A
    1          917            A
    2          920            A
    3          727            A
    4          919            A
    5          555            A
    6          552            A
    7          551            A
    8          846            A
    9          845            A
    10         849            A
    11         913            A
    12         706            A
    13

Positive preds. above 0.5 probability (average).

In [29]:
alloses_results = {
    pdb: {
        pocket: {
            "pred": 1 if pocketd["pro_ave"] >= 0.5 else 0,
            **pocketd
        }
        for pocket, pocketd in pockets.items()
    }
    for pdb, pockets in alloses_results.items()
}
alloses_results

{'7gqu': {'pocket7': {'pred': 1,
   'pro_ave': 0.6077189166132947,
   'residues':    auth_seq_id auth_asym_id
   0          601            A
   1          829            A
   2          573            A
   3          577            A
   4          578            A
   ..         ...          ...
   73         923            A
   74         551            A
   75         547            A
   76         883            A
   77         801            A
   
   [78 rows x 2 columns]},
  'pocket20': {'pred': 0,
   'pro_ave': 0.4671694797381203,
   'residues':    auth_seq_id auth_asym_id
   0          916            A
   1          917            A
   2          920            A
   3          727            A
   4          919            A
   5          555            A
   6          552            A
   7          551            A
   8          846            A
   9          845            A
   10         849            A
   11         913            A
   12         706            A
   13       

## MEF-AlloSite

In [30]:
import json

Top 3 pockets for each PDB are predicted positive.

In [31]:
mefallosite_resultsf = "MEF-AlloSite/mef-allosite_results.json"

with open(mefallosite_resultsf, "r") as f:
    mefallosite_results = json.load(f)

len(mefallosite_results), mefallosite_results

(9,
 {'7gqu': {'pocket15': {'prob': 0.07944739485780399,
    'pred': 0,
    'residues': {'auth_asym_id': ['A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A'],
     'auth_seq_id': ['760',
      '762',
      '766',
      '790',
      '813',
      '814',
      '815',
      '816',
      '832'],
     'pdbx_PDB_ins_code': ['?', '?', '?', '?', '?', '?', '?', '?', '?']}},
   'pocket21': {'prob': 0.09905595394472282,
    'pred': 0,
    'residues': {'auth_asym_id': ['A', 'A', 'A', 'A', 'A', 'A', 'A', 'A'],
     'auth_seq_id': ['561', '564', '588', '590', '657', '661', '662', '663'],
     'pdbx_PDB_ins_code': ['?', '?', '?', '?', '?', '?', '?', '?']}},
   'pocket22': {'prob': 0.20237085223197937,
    'pred': 0,
    'residues': {'auth_asym_id': ['A',
      'A',
      'A',
      'A',
      'A',
      'A',
      'A',
      'A',
      'A',
      'A',
      'A',
      'A',
      'A'],
     'auth_seq_id': ['740',
      '752',
      '753',
      '763',
      '764',
      '765',
      '767',
      '834',
      

In [32]:
mefallosite_results = {
    pdb: {
        pocket: {
            k: v if k != "residues" else pd.DataFrame(v)
            for k, v in pocketd.items()
        }
        for pocket, pocketd in pockets.items()
    } 
    for pdb, pockets in mefallosite_results.items()
}

mefallosite_results

{'7gqu': {'pocket15': {'prob': 0.07944739485780399,
   'pred': 0,
   'residues':   auth_asym_id auth_seq_id pdbx_PDB_ins_code
   0            A         760                 ?
   1            A         762                 ?
   2            A         766                 ?
   3            A         790                 ?
   4            A         813                 ?
   5            A         814                 ?
   6            A         815                 ?
   7            A         816                 ?
   8            A         832                 ?},
  'pocket21': {'prob': 0.09905595394472282,
   'pred': 0,
   'residues':   auth_asym_id auth_seq_id pdbx_PDB_ins_code
   0            A         561                 ?
   1            A         564                 ?
   2            A         588                 ?
   3            A         590                 ?
   4            A         657                 ?
   5            A         661                 ?
   6            A         662     

## ALLO

Top pocket is taken as positive prediction.

In [33]:
allo_resultsf = "ALLO/ALLO_results.json"

with open(allo_resultsf, "r") as f:
    allo_results = json.load(f)

len(allo_results), allo_results

(9,
 {'9dnm': {'P_10': {'pred': 0,
    'prob': 0.04074,
    'residues': {'pdbx_PDB_ins_code': ['?',
      '?',
      '?',
      '?',
      '?',
      '?',
      '?',
      '?',
      '?',
      '?',
      '?'],
     'auth_asym_id': ['A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A'],
     'auth_seq_id': ['58',
      '59',
      '60',
      '62',
      '76',
      '78',
      '143',
      '144',
      '145',
      '147',
      '165']}},
   'P_11': {'pred': 0,
    'prob': 0.04071,
    'residues': {'pdbx_PDB_ins_code': ['?',
      '?',
      '?',
      '?',
      '?',
      '?',
      '?',
      '?',
      '?'],
     'auth_asym_id': ['A', 'A', 'A', 'A', 'A', 'A', 'A', 'A', 'A'],
     'auth_seq_id': ['341',
      '342',
      '343',
      '344',
      '345',
      '347',
      '348',
      '413',
      '414']}},
   'P_12': {'pred': 0,
    'prob': 0.04063,
    'residues': {'pdbx_PDB_ins_code': ['?',
      '?',
      '?',
      '?',
      '?',
      '?',
      '?',
      '?',
      '?'],

In [34]:
allo_results = {
    pdb: {
        pocket: {
            k: v if k != "residues" else pd.DataFrame(v)
            for k, v in pocketd.items()
        }
        for pocket, pocketd in pockets.items()
    } 
    for pdb, pockets in allo_results.items()
}

allo_results

{'9dnm': {'P_10': {'pred': 0,
   'prob': 0.04074,
   'residues':    pdbx_PDB_ins_code auth_asym_id auth_seq_id
   0                  ?            A          58
   1                  ?            A          59
   2                  ?            A          60
   3                  ?            A          62
   4                  ?            A          76
   5                  ?            A          78
   6                  ?            A         143
   7                  ?            A         144
   8                  ?            A         145
   9                  ?            A         147
   10                 ?            A         165},
  'P_11': {'pred': 0,
   'prob': 0.04071,
   'residues':   pdbx_PDB_ins_code auth_asym_id auth_seq_id
   0                 ?            A         341
   1                 ?            A         342
   2                 ?            A         343
   3                 ?            A         344
   4                 ?            A         345
   5  

## All models

In [27]:
defaults = { "all_pockets_in_output": True, "prob_key": "prob" }

models = {
    "model5": { "results": model5_results, **defaults, "labelling": None },
    
    "allositepro": { "results": allositepro_results, "all_pockets_in_output": False, "prob_key": "hitScore" },
    # "stingallo": { "results": stingallo_results, "all_pockets_in_output": False, "prob_key": None },
    # "allofusion": { "results": allofusion_results, "all_pockets_in_output": False, "prob_key": None },
    
    # "passer_ensemble": { "results": passer_results["ensemble"], **defaults, "prob_key": "prob/score" },
    # "passer_automl": { "results": passer_results["automl"], **defaults, "prob_key": "prob/score" },
    # "passer_rank": { "results": passer_results["rank"], **defaults, "prob_key": "prob/score" },
    # "deepallo": { "results": deepallo_results, **defaults },
    # "alloses": { "results": alloses_results, **defaults, "prob_key": "pro_ave" },
    # "mefallosite": { "results": mefallosite_results, **defaults },
    # "allo": { "results": allo_results, **defaults },
}
models

{'model5': {'results': {'22mj': {'pocket11': {'prob': 0.015926241874694824,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket7': {'prob': 0.0002722057106439024,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket1': {'prob': 5.723202775698155e-06,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket2': {'prob': 0.80405592918396,
     'pred': 1,
     'label': 1,
     'max_overlap': 0.9285714285714286,
     'pocket_in_site': 0.5909090909090909,
     'site_in_pocket': 0.9285714285714286},
    'pocket4': {'prob': 2.3296684958040714e-05,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.0,
     'pocket_in_site': 0.0,
     'site_in_pocket': 0.0},
    'pocket9': {'prob': 0.000627053901553154,
     'pred': 0,
     'label': 0,
     'max_overlap': 0.08333333333333333,


# Labelling

In [17]:
# Percentage of residues of "one" in "other"
get_overlap = lambda one, other: (
    len( one.merge(other) ) / len(one)
)

def get_label(overlaps, site_in_pocket=None, pocket_in_site=None):
    assert not (site_in_pocket==None and pocket_in_site==None)
    
    if site_in_pocket is None:
        return int( round(overlaps["pocket_in_site"], 2) >= pocket_in_site )
    if pocket_in_site is None:
        return int( round(overlaps["site_in_pocket"], 2) >= site_in_pocket )
    return int( round(overlaps["site_in_pocket"], 2) >= site_in_pocket or round(overlaps["pocket_in_site"], 2) >= pocket_in_site )

In [18]:
get_overlaps = lambda pdb, pocketd: {
    name: get_overlap(*one_in_other) 
        for site in news_sites[pdb] 
            for name, one_in_other in (
                ("pocket_in_site", (pocketd["residues"], site["site"])),
                ("site_in_pocket", (site["site"], pocketd["residues"])),
            )
}

get_overlaps_special = lambda pdb, pocketd: {
    name: get_overlap(*one_in_other) 
        for pocketres in (pocketd["residues"],)
            for sited in news_sites[pdb]
                for site in (sited["site"].query(f"auth_asym_id == '{pocketres.auth_asym_id.unique().item()}'"),)
                    for name, one_in_other in (
                        ("pocket_in_site", (pocketres, site)),
                        ("site_in_pocket", (site, pocketres)),
                    )
}

In [19]:
def label_results(resultsd, site_in_pocket=0.65, pocket_in_site=None, prob_key=None, topx=False, special_8aq6_sc=False):
    df = pd.DataFrame((
        {
            "pdb": pdb,
            "pocket": pocket,
            **{"prob": pocketd[prob_key] for prob_key in (prob_key,) if prob_key is not None},
            "pred": pocketd["pred"],
            "label": get_label(overlaps, site_in_pocket, pocket_in_site),
            "max_overlap": max(overlaps.values()),
            **overlaps,
        }
        for pdb, pockets in resultsd.items()
        for pocket, pocketd in pockets.items()
        for overlaps in ((
            get_overlaps_special(pdb, pocketd)
            if special_8aq6_sc and pdb == "8aq6"
            else get_overlaps(pdb, pocketd)
        ),)
    ))
    if prob_key is not None and topx:
        df["pred"] = (
            # Start from a Series where each value/row (sample/pocket) is the total number of pos. labels on its PDB
            df.groupby("pdb")["label"].transform("sum")
            # Then subtract this "total num. of pos. in a PDB" by the rank of each pocket in a PDB, sorted by the probability
            .sub(df.groupby("pdb")["prob"].rank(method="first", ascending=False))
            # If the subtraction is positive or 0 it means that the pocket is in the topX and will be assigned 1
            >= 0
        ).astype(int)
        # If a PDB has no + labelled pocket, assign the highest prob. as positive
        for pdb, group in df.groupby("pdb"):
            if group["pred"].sum() == 0:
                df.loc[ group["prob"].idxmax(), "pred" ] = 1
            
    
    return df.sort_values("max_overlap", ascending=False)

## Model 5.

In [20]:
def label_results_ours(resultsd, site_in_pocket=0.65, pocket_in_site=None, prob_key=None, topx=False, special_8aq6_sc=False):
    df = pd.DataFrame((
        {
            "pdb": pdb,
            "pocket": pocket,
            **{"prob": pocketd[prob_key] for prob_key in (prob_key,) if prob_key is not None},
            "pred": pocketd["pred"],
            "label": get_label(overlaps, site_in_pocket, pocket_in_site),
            "max_overlap": max(overlaps.values()),
            **overlaps,
        }
        for pdb, pockets in resultsd.items()
        for pocket, pocketd in pockets.items()
        for overlaps in ({
            "pocket_in_site": pocketd["pocket_in_site"],
            "site_in_pocket": pocketd["site_in_pocket"],
        },)
    ))
    if prob_key is not None and topx:
        df["pred"] = (
            # Start from a Series where each value/row (sample/pocket) is the total number of pos. labels on its PDB
            df.groupby("pdb")["label"].transform("sum")
            # Then subtract this "total num. of pos. in a PDB" by the rank of each pocket in a PDB, sorted by the probability
            .sub(df.groupby("pdb")["prob"].rank(method="first", ascending=False))
            # If the subtraction is positive or 0 it means that the pocket is in the topX and will be assigned 1
            >= 0
        ).astype(int)
        # If a PDB has no + labelled pocket, assign the highest prob. as positive
        for pdb, group in df.groupby("pdb"):
            if group["pred"].sum() == 0:
                df.loc[ group["prob"].idxmax(), "pred" ] = 1
            
    
    return df.sort_values("max_overlap", ascending=False)

In [21]:
our_results = label_results_ours(model5_results, site_in_pocket=0.65, pocket_in_site=None, prob_key="prob", topx=True, special_8aq6_sc=False)

our_results.sort_values(["pred", "label", "prob", "site_in_pocket"], ascending=False).iloc[:40]

,pdb,pocket,prob,pred,label,max_overlap,pocket_in_site,site_in_pocket
657,9ntc,pocket1,0.982443,1,1,0.717391,0.717391,0.647059
532,8v81,pocket1,0.977954,1,1,0.962963,0.208000,0.962963
673,9oul,pocket1,0.950287,1,1,0.903226,0.259259,0.903226
498,8uk6,pocket1,0.946666,1,1,0.952381,0.540541,0.952381
30,6vvq,pocket1,0.929778,1,1,0.777778,0.777778,0.736842
326,8f4s,pocket1,0.923800,1,1,0.846154,0.733333,0.846154
475,8qni,pocket1,0.917425,1,1,0.966667,0.322222,0.966667
21,6s3a,pocket1,0.916088,1,1,0.740741,0.512821,0.740741
449,8jp0,pocket1,0.906662,1,1,0.781250,0.320513,0.781250
146,7pt0,pocket2,0.900742,1,1,0.863636,0.527778,0.863636


In [22]:
our_results.sort_values(["pred", "label", "pdb"], ascending=False).iloc[:40]

,pdb,pocket,prob,pred,label,max_overlap,pocket_in_site,site_in_pocket
692,9ow9,pocket1,0.775315,1,1,0.916667,0.611111,0.916667
673,9oul,pocket1,0.950287,1,1,0.903226,0.259259,0.903226
657,9ntc,pocket1,0.982443,1,1,0.717391,0.717391,0.647059
608,9fsj,pocket1,0.502134,1,1,0.812500,0.812500,0.764706
592,9ebs,pocket7,0.796260,1,1,0.857143,0.500000,0.857143
532,8v81,pocket1,0.977954,1,1,0.962963,0.208000,0.962963
498,8uk6,pocket1,0.946666,1,1,0.952381,0.540541,0.952381
475,8qni,pocket1,0.917425,1,1,0.966667,0.322222,0.966667
449,8jp0,pocket1,0.906662,1,1,0.781250,0.320513,0.781250
405,8hx9,pocket1,0.900688,1,1,0.884615,0.469388,0.884615


In [23]:
our_results.query("pdb == '9mft'").sort_values(["prob"], ascending=False).iloc[:40]

,pdb,pocket,prob,pred,label,max_overlap,pocket_in_site,site_in_pocket
637,9mft,pocket1,9.966699e-01,1,0,0.000000,0.000000,0.000000
631,9mft,pocket7,7.835469e-01,0,0,0.000000,0.000000,0.000000
639,9mft,pocket2,4.122567e-01,0,0,0.000000,0.000000,0.000000
650,9mft,pocket3,1.622391e-01,0,0,0.000000,0.000000,0.000000
640,9mft,pocket20,5.491935e-02,0,0,0.000000,0.000000,0.000000
627,9mft,pocket18,4.151691e-02,0,1,0.791667,0.791667,0.730769
654,9mft,pocket6,1.697234e-02,0,0,0.076923,0.068966,0.076923
633,9mft,pocket13,7.605394e-03,0,0,0.000000,0.000000,0.000000
643,9mft,pocket30,5.665230e-03,0,0,0.000000,0.000000,0.000000
653,9mft,pocket21,2.508157e-03,0,0,0.000000,0.000000,0.000000


In [26]:
our_results.query("pdb == '8e24'").sort_values(["prob"], ascending=False).iloc[:40]

,pdb,pocket,prob,pred,label,max_overlap,pocket_in_site,site_in_pocket
309,8e24,pocket1,0.494149,1,0,0.166667,0.147059,0.166667
313,8e24,pocket4,0.412652,0,1,0.888889,0.888889,0.800000
303,8e24,pocket7,0.069771,0,0,0.000000,0.000000,0.000000
305,8e24,pocket13,0.045207,0,0,0.000000,0.000000,0.000000
315,8e24,pocket9,0.023964,0,0,0.000000,0.000000,0.000000
322,8e24,pocket19,0.015794,0,0,0.000000,0.000000,0.000000
321,8e24,pocket12,0.002156,0,0,0.000000,0.000000,0.000000
324,8e24,pocket6,0.001389,0,0,0.062500,0.062500,0.033333
320,8e24,pocket3,0.000987,0,0,0.000000,0.000000,0.000000
302,8e24,pocket15,0.000968,0,0,0.000000,0.000000,0.000000


In [9]:
our_results.query("pdb == '7oy0'").sort_values(["prob"], ascending=False).iloc[:40]

NameError: name 'our_results' is not defined

In [27]:
our_results.query("pdb == '8jb3'").sort_values(["prob"], ascending=False).iloc[:40]

,pdb,pocket,prob,pred,label,max_overlap,pocket_in_site,site_in_pocket
426,8jb3,pocket1,8.276858e-01,1,0,0.000000,0.000000,0.00
433,8jb3,pocket3,2.922507e-01,0,1,0.920000,0.851852,0.92
427,8jb3,pocket2,2.179353e-01,0,0,0.000000,0.000000,0.00
430,8jb3,pocket10,7.053933e-02,0,0,0.000000,0.000000,0.00
423,8jb3,pocket7,3.764052e-02,0,0,0.000000,0.000000,0.00
421,8jb3,pocket11,1.984207e-03,0,0,0.000000,0.000000,0.00
428,8jb3,pocket4,1.827343e-03,0,0,0.058824,0.058824,0.04
435,8jb3,pocket6,7.793623e-04,0,0,0.411765,0.411765,0.28
420,8jb3,pocket16,6.153825e-04,0,0,0.000000,0.000000,0.00
432,8jb3,pocket8,2.331167e-04,0,0,0.000000,0.000000,0.00


In [28]:
our_results.query("pdb == '9b96'").sort_values(["prob"], ascending=False).iloc[:40]

,pdb,pocket,prob,pred,label,max_overlap,pocket_in_site,site_in_pocket
570,9b96,pocket1,7.148713e-01,1,0,0.000000,0.000000,0.000000
571,9b96,pocket2,5.042924e-01,0,0,0.000000,0.000000,0.000000
583,9b96,pocket6,1.270476e-01,0,0,0.133333,0.105263,0.133333
568,9b96,pocket23,1.180363e-02,0,0,0.000000,0.000000,0.000000
582,9b96,pocket21,5.604086e-03,0,1,0.833333,0.833333,0.666667
579,9b96,pocket3,5.163046e-03,0,0,0.000000,0.000000,0.000000
564,9b96,pocket15,3.829577e-03,0,0,0.000000,0.000000,0.000000
575,9b96,pocket9,1.604969e-03,0,0,0.000000,0.000000,0.000000
577,9b96,pocket5,1.177435e-03,0,0,0.000000,0.000000,0.000000
565,9b96,pocket7,7.283384e-04,0,0,0.000000,0.000000,0.000000


In [29]:
our_results.query("pdb == '9d2r'").sort_values(["prob"], ascending=False).iloc[:40]

,pdb,pocket,prob,pred,label,max_overlap,pocket_in_site,site_in_pocket
589,9d2r,pocket3,0.776362,1,0,0.000000,0.000000,0.000000
585,9d2r,pocket1,0.554641,0,1,0.941176,0.695652,0.941176
587,9d2r,pocket4,0.324540,0,0,0.000000,0.000000,0.000000
586,9d2r,pocket2,0.067344,0,0,0.117647,0.111111,0.117647
588,9d2r,pocket5,0.038538,0,0,0.000000,0.000000,0.000000
584,9d2r,pocket7,0.006982,0,0,0.181818,0.181818,0.117647
590,9d2r,pocket6,0.000254,0,0,0.000000,0.000000,0.000000


## AllositePro

In [25]:
label_results(allositepro_results, site_in_pocket=0.65, pocket_in_site=0.6)

,pdb,pocket,pred,label,max_overlap,pocket_in_site,site_in_pocket
8,8crc,pocket1,1,1,1.000000,1.000000,0.692308
2,6vvq,pocket0,1,1,1.000000,1.000000,0.684211
10,8jp0,pocket0,1,1,1.000000,1.000000,0.312500
13,8uk6,pocket1,1,1,0.952381,0.833333,0.952381
4,6z1m,pocket1,1,1,0.944444,0.944444,0.472222
11,8qni,pocket0,1,1,0.933333,0.608696,0.933333
15,9fsj,pocket0,1,1,0.923077,0.923077,0.705882
9,8f4s,pocket0,1,1,0.846154,0.733333,0.846154
0,22mj,pocket0,1,1,0.821429,0.766667,0.821429
16,9oul,pocket0,1,1,0.814815,0.814815,0.709677


In [28]:
models["allositepro"]["labelling"] = {"site_in_pocket": 0.65, "pocket_in_site": 0.6}

## PASSer

In [41]:
label_results(passer_results["ensemble"], site_in_pocket=0.65, pocket_in_site=0.7).iloc[:40]

,pdb,pocket,pred,label,max_overlap,pocket_in_site,site_in_pocket
262,8uk6,1,1,1,0.904762,0.904762,0.904762
306,8v81,49,1,1,0.888889,0.666667,0.888889
241,8qni,13,1,1,0.833333,0.510204,0.833333
0,7gqu,20,1,1,0.758621,0.687500,0.758621
141,8aq6,8,1,1,0.740741,0.740741,0.689655
158,8f4s,9,1,1,0.730769,0.730769,0.730769
182,8jp0,57,1,1,0.718750,0.534884,0.718750
31,7yg5,29,0,1,0.708333,0.708333,0.586207
166,8f4s,15,0,0,0.625000,0.625000,0.192308
250,8qni,1,0,0,0.428571,0.428571,0.200000


In [42]:
models["passer_ensemble"]["labelling"] = {"site_in_pocket": 0.65, "pocket_in_site": 0.7}

In [43]:
label_results(passer_results["automl"], site_in_pocket=0.65, pocket_in_site=0.7).iloc[:40]

,pdb,pocket,pred,label,max_overlap,pocket_in_site,site_in_pocket
262,8uk6,1,1,1,0.904762,0.904762,0.904762
307,8v81,49,0,1,0.888889,0.666667,0.888889
241,8qni,13,1,1,0.833333,0.510204,0.833333
1,7gqu,20,1,1,0.758621,0.687500,0.758621
141,8aq6,8,1,1,0.740741,0.740741,0.689655
158,8f4s,9,1,1,0.730769,0.730769,0.730769
182,8jp0,57,1,1,0.718750,0.534884,0.718750
31,7yg5,29,0,1,0.708333,0.708333,0.586207
164,8f4s,15,0,0,0.625000,0.625000,0.192308
149,8aq6,9,0,0,0.428571,0.428571,0.103448


In [44]:
models["passer_automl"]["labelling"] = {"site_in_pocket": 0.65, "pocket_in_site": 0.7}

In [45]:
label_results(passer_results["rank"], site_in_pocket=0.65, pocket_in_site=0.7).iloc[:40]

,pdb,pocket,pred,label,max_overlap,pocket_in_site,site_in_pocket
262,8uk6,1,1,1,0.904762,0.904762,0.904762
325,8v81,49,0,1,0.888889,0.666667,0.888889
249,8qni,13,0,1,0.833333,0.510204,0.833333
3,7gqu,20,0,1,0.758621,0.687500,0.758621
144,8aq6,8,0,1,0.740741,0.740741,0.689655
160,8f4s,9,1,1,0.730769,0.730769,0.730769
198,8jp0,57,0,1,0.718750,0.534884,0.718750
80,7yg5,29,0,1,0.708333,0.708333,0.586207
177,8f4s,15,0,0,0.625000,0.625000,0.192308
147,8aq6,9,0,0,0.428571,0.428571,0.103448


In [46]:
models["passer_rank"]["labelling"] = {"site_in_pocket": 0.65, "pocket_in_site": 0.7}

## DeepAllo

In [47]:
label_results(deepallo_results, site_in_pocket=0.65, pocket_in_site=0.65, special_8aq6_sc=True).iloc[:40]

,pdb,pocket,pred,label,max_overlap,pocket_in_site,site_in_pocket
245,8uk6,pocket1,1,1,0.904762,0.904762,0.904762
288,8v81,pocket2,1,1,0.888889,0.666667,0.888889
224,8qni,pocket1,1,1,0.833333,0.543478,0.833333
1,7gqu,pocket2,1,1,0.758621,0.687500,0.758621
140,8f4s,pocket1,1,1,0.730769,0.730769,0.730769
167,8jp0,pocket1,0,1,0.718750,0.534884,0.718750
31,7yg5,pocket2,0,1,0.708333,0.708333,0.586207
396,8aq6,pocket1_G,1,1,0.666667,0.666667,0.375000
153,8f4s,pocket13,0,0,0.625000,0.625000,0.192308
223,8qni,pocket2,1,0,0.428571,0.428571,0.200000


In [48]:
models["deepallo"]["labelling"] = {"site_in_pocket": 0.65, "pocket_in_site": 0.65, "special_8aq6_sc": True}

## STINGAllo

In [49]:
label_results(stingallo_results, site_in_pocket=None, pocket_in_site=1)

,pdb,pocket,pred,label,max_overlap,pocket_in_site,site_in_pocket
0,8qni,pocket,1,1,1.0,1.0,0.033333
2,8aq6,pocket,1,1,1.0,1.0,0.034483
1,7gqu,pocket,1,0,0.0,0.0,0.000000


In [50]:
models["stingallo"]["labelling"] = {"site_in_pocket": None, "pocket_in_site": 1}

## AlloFusion

In [51]:
result = label_results(allofusion_results, site_in_pocket=None, pocket_in_site=1)
result

,pdb,pocket,pred,label,max_overlap,pocket_in_site,site_in_pocket
54,8aq6,pocket_G_10,1,1,1.0,1.0,0.034483
19,8qni,pocket_A_141,1,1,1.0,1.0,0.033333
22,8uk6,pocket_A_265,1,1,1.0,1.0,0.047619
23,8uk6,pocket_A_268,1,1,1.0,1.0,0.047619
57,8aq6,pocket_G_57,1,1,1.0,1.0,0.034483
...,...,...,...,...,...,...,...
24,8uk6,pocket_A_465,1,0,0.0,0.0,0.000000
21,8qni,pocket_A_289,1,0,0.0,0.0,0.000000
20,8qni,pocket_A_143,1,0,0.0,0.0,0.000000
18,8qni,pocket_A_132,1,0,0.0,0.0,0.000000


In [52]:
mask = ((result.pred == 1) & (result.label == 1))
mask

54     True
19     True
22     True
23     True
57     True
      ...  
24    False
21    False
20    False
18    False
30    False
Length: 74, dtype: bool

In [53]:
pd.concat((
    result.loc[result[mask].drop_duplicates("pdb").index],
    result[~mask]
))

,pdb,pocket,pred,label,max_overlap,pocket_in_site,site_in_pocket
54,8aq6,pocket_G_10,1,1,1.0,1.0,0.034483
19,8qni,pocket_A_141,1,1,1.0,1.0,0.033333
22,8uk6,pocket_A_265,1,1,1.0,1.0,0.047619
8,7gqu,pocket_A_891,1,1,1.0,1.0,0.034483
28,8v81,pocket_A_347,1,1,1.0,1.0,0.037037
...,...,...,...,...,...,...,...
24,8uk6,pocket_A_465,1,0,0.0,0.0,0.000000
21,8qni,pocket_A_289,1,0,0.0,0.0,0.000000
20,8qni,pocket_A_143,1,0,0.0,0.0,0.000000
18,8qni,pocket_A_132,1,0,0.0,0.0,0.000000


In [54]:
models["allofusion"]["labelling"] = {"site_in_pocket": None, "pocket_in_site": 1}

## AllosES

In [55]:
label_results(alloses_results, site_in_pocket=0.65, pocket_in_site=0.7, special_8aq6_sc=True).iloc[:40]

,pdb,pocket,pred,label,max_overlap,pocket_in_site,site_in_pocket
244,8uk6,pocket1,0,1,0.904762,0.904762,0.904762
287,8v81,pocket49,0,1,0.888889,0.666667,0.888889
222,8qni,pocket13,0,1,0.833333,0.510204,0.833333
1,7gqu,pocket20,0,1,0.758621,0.687500,0.758621
138,8f4s,pocket9,1,1,0.730769,0.730769,0.730769
162,8jp0,pocket57,1,1,0.718750,0.534884,0.718750
45,7yg5,pocket29,0,1,0.708333,0.708333,0.586207
397,8aq6,pocket1_G,0,0,0.666667,0.666667,0.375000
152,8f4s,pocket15,0,0,0.625000,0.625000,0.192308
223,8qni,pocket1,0,0,0.428571,0.428571,0.200000


In [56]:
models["alloses"]["labelling"] = {"site_in_pocket": 0.65, "pocket_in_site": 0.65, "special_8aq6_sc": True}

## MEF-AlloSite

In [57]:
label_results(mefallosite_results, site_in_pocket=0.65, pocket_in_site=0.7).iloc[:40]

,pdb,pocket,pred,label,max_overlap,pocket_in_site,site_in_pocket
243,8uk6,pocket1,0,1,0.904762,0.904762,0.904762
287,8v81,pocket36,0,1,0.888889,0.666667,0.888889
209,8qni,pocket13,0,1,0.833333,0.510204,0.833333
17,7gqu,pocket14,1,1,0.758621,0.687500,0.758621
114,8aq6,pocket5,1,1,0.740741,0.740741,0.689655
133,8f4s,pocket8,1,1,0.720000,0.720000,0.692308
184,8jp0,pocket49,0,1,0.718750,0.534884,0.718750
55,7yg5,pocket12,0,1,0.708333,0.708333,0.586207
126,8f4s,pocket16,0,0,0.625000,0.625000,0.192308
6,7gqu,pocket3,0,0,0.454545,0.454545,0.172414


In [58]:
models["mefallosite"]["labelling"] = {"site_in_pocket": 0.65, "pocket_in_site": 0.7}

## ALLO

In [59]:
label_results(allo_results, site_in_pocket=0.65, pocket_in_site=None).iloc[:40]

,pdb,pocket,pred,label,max_overlap,pocket_in_site,site_in_pocket
42,8uk6,P_0,0,1,1.000000,0.375000,1.000000
22,8qni,P_0,1,1,1.000000,0.312500,1.000000
197,8jp0,P_0,1,1,0.968750,0.248000,0.968750
176,7gqu,P_1,0,1,0.950000,0.950000,0.655172
163,8v81,P_0,1,1,0.925926,0.500000,0.925926
105,8aq6,P_0,0,1,0.862069,0.510204,0.862069
120,8f4s,P_0,0,1,0.846154,0.758621,0.846154
12,9dnm,P_0,1,1,0.700000,0.066038,0.700000
93,7yg5,P_1,0,1,0.655172,0.513514,0.655172
65,7yg5,P_39,0,0,0.500000,0.500000,0.137931


In [60]:
models["allo"]["labelling"] = {"site_in_pocket": 0.65, "pocket_in_site": None}

<br>

In [61]:
pd.to_pickle(models, "models.pkl")

# Scoring

## Original

In [70]:
models_preds = {}

for model, modeld in models.items():
    results = modeld["results"]
    if modeld["labelling"] is not None:
        labelled_results = label_results(results, **modeld["labelling"], prob_key=modeld["prob_key"])
    else:
        labelled_results = pd.DataFrame((
            {
                "pdb": pdb,
                "pocket": pocket,
                **pocketd
            }
            for pdb, pockets in results.items()
            for pocket, pocketd in pockets.items()
        ))

    preds = {}
    for pdb in news:
        if pdb == "9dnm": continue #######################################################################################
        pdbpreds = labelled_results.query(f"pdb == '{pdb}'")
        total = len(pdbpreds)
        if total > 0:
            # Top1 pred for our model
            if model == "model5":
                # pdbpreds.loc[:, "pred"] = pdbpreds[["prob"]].apply(lambda x: (x == x.max()).astype(int)).values
                pass
            elif model == "allofusion":
                pdbpreds = pdbpreds.drop(
                    pdbpreds.loc[lambda x: (x.pred == 1) & (x.label == 1)]
                    .index[1:]
                )
            tp = len( pdbpreds.loc[lambda x: (x["pred"] == 1) & (x["label"] == 1)] )
            fp = len( pdbpreds.loc[lambda x: (x["pred"] == 1) & (x["label"] == 0)] )
            fn = len( pdbpreds.loc[lambda x: (x["pred"] == 0) & (x["label"] == 1)] )
            if tp + fn == 0:
                fn = 1 # There's at least 1 allo. pocket per PDB
        else:
            tp, fp = 0, 0
            fn = 1 # There's at least 1 allo. pocket per PDB

        if total == 0 or not modeld["all_pockets_in_output"]:
            preds[pdb] = {
                "tp": tp,
                "fn": fn,
                "fp": fp
            }
        else:
            preds[pdb] = {
                "total": total,
                "tp": tp,
                "fn": fn,
                "fp": fp
            }

    models_preds[model] = preds

models_preds

{'model5': {'7gqu': {'total': 15, 'tp': 1, 'fn': 0, 'fp': 0},
  '7yg5': {'total': 66, 'tp': 0, 'fn': 1, 'fp': 1},
  '8aq6': {'total': 13, 'tp': 1, 'fn': 0, 'fp': 0},
  '8f4s': {'total': 10, 'tp': 1, 'fn': 0, 'fp': 0},
  '8jp0': {'total': 34, 'tp': 1, 'fn': 0, 'fp': 0},
  '8qni': {'total': 17, 'tp': 1, 'fn': 0, 'fp': 0},
  '8uk6': {'total': 30, 'tp': 1, 'fn': 0, 'fp': 0},
  '8v81': {'total': 45, 'tp': 1, 'fn': 0, 'fp': 2}},
 'allositepro': {'7gqu': {'tp': 1, 'fn': 0, 'fp': 0},
  '7yg5': {'tp': 0, 'fn': 1, 'fp': 1},
  '8aq6': {'tp': 1, 'fn': 0, 'fp': 1},
  '8f4s': {'tp': 1, 'fn': 0, 'fp': 0},
  '8jp0': {'tp': 1, 'fn': 0, 'fp': 0},
  '8qni': {'tp': 1, 'fn': 0, 'fp': 0},
  '8uk6': {'tp': 1, 'fn': 0, 'fp': 1},
  '8v81': {'tp': 1, 'fn': 0, 'fp': 0}},
 'stingallo': {'7gqu': {'tp': 0, 'fn': 1, 'fp': 1},
  '7yg5': {'tp': 0, 'fn': 1, 'fp': 0},
  '8aq6': {'tp': 1, 'fn': 0, 'fp': 0},
  '8f4s': {'tp': 0, 'fn': 1, 'fp': 0},
  '8jp0': {'tp': 0, 'fn': 1, 'fp': 0},
  '8qni': {'tp': 1, 'fn': 0, 'fp': 0}

In [29]:
from sklearn.metrics import matthews_corrcoef, f1_score, confusion_matrix

In [72]:
models_metrics = {}

for model, preds in models_preds.items():
    df = pd.DataFrame(preds).T
    
    if models[model]["all_pockets_in_output"]:
        y_true = [1] * df["tp"].sum() + [1] * df["fn"].sum() + [0] * df["fp"].sum() + [0] * (df["total"].sum() - df[["tp", "fn", "fp"]].sum().sum())
        y_pred = [1] * df["tp"].sum() + [0] * df["fn"].sum() + [1] * df["fp"].sum() + [0] * (df["total"].sum() - df[["tp", "fn", "fp"]].sum().sum())

        models_metrics[model] = {
            "model": model,
            "tp": df["tp"].sum(),
            "fn": df["fn"].sum(),
            "fp": df["fp"].sum(),
            "mcc": matthews_corrcoef(y_true, y_pred),
            "macro-f1": f1_score(y_true, y_pred, average="macro"),
            "confmat": pd.DataFrame(confusion_matrix(y_true, y_pred))
        }
        
    else:
        models_metrics[model] = {
            "model": model,
            "tp": df["tp"].sum(),
            "fn": df["fn"].sum(),
            "fp": df["fp"].sum(),
        }

models_metrics = pd.DataFrame(models_metrics).T.sort_values(["tp", "mcc"], ascending=False)
models_metrics

,model,tp,fn,fp,mcc,macro-f1,confmat
model5,model5,7,1,3,0.774031,0.884364,0 1 0 219 3 1 1 7
passer_ensemble,passer_ensemble,7,1,17,0.489352,0.706319,0 1 0 353 17 1 1 7
allositepro,allositepro,7,1,3,NaN,NaN,NaN
deepallo,deepallo,6,2,18,0.414027,0.673726,0 1 0 353 18 1 2 6
passer_automl,passer_automl,6,2,18,0.413975,0.673688,0 1 0 352 18 1 2 6
allofusion,allofusion,5,3,56,NaN,NaN,NaN
mefallosite,mefallosite,3,5,5,0.358553,0.679276,0 1 0 299 5 1 5 3
allo,allo,3,5,5,0.346751,0.673376,0 1 0 172 5 1 5 3
alloses,alloses,2,6,3,0.304708,0.647773,0 1 0 366 3 1 6 2
passer_rank,passer_rank,2,6,22,0.112467,0.543163,0 1 0 348 22 1 6 2


In [65]:
models_metrics.infer_objects().to_csv("models_metrics.csv", index=False, sep="\t", decimal=",")

## TopX

In [30]:
models_preds = {}

for model, modeld in models.items():
    results = modeld["results"]
    if modeld["labelling"] is not None:
        labelled_results = label_results(results, **modeld["labelling"], prob_key=modeld["prob_key"], topx=True)
    else:
        labelled_results = pd.DataFrame((
            {
                "pdb": pdb,
                "pocket": pocket,
                **pocketd
            }
            for pdb, pockets in results.items()
            for pocket, pocketd in pockets.items()
        ))

    preds = {}
    for pdb in news:
        if pdb == "9dnm": continue #######################################################################################
        pdbpreds = labelled_results.query(f"pdb == '{pdb}'")
        total = len(pdbpreds)
        if total > 0:
            # Top1 pred for our model
            if model == "model5":
                pdbpreds.loc[:, "pred"] = pdbpreds[["prob"]].apply(lambda x: (x == x.max()).astype(int)).values
            # elif model == "allofusion":
            #     pdbpreds = pdbpreds.drop(
            #         pdbpreds.loc[lambda x: (x.pred == 1) & (x.label == 1)]
            #         .index[1:]
            #     )
            tp = len( pdbpreds.loc[lambda x: (x["pred"] == 1) & (x["label"] == 1)] )
            fp = len( pdbpreds.loc[lambda x: (x["pred"] == 1) & (x["label"] == 0)] )
            fn = len( pdbpreds.loc[lambda x: (x["pred"] == 0) & (x["label"] == 1)] )
            if tp + fn == 0:
                fn = 1 # There's at least 1 allo. pocket per PDB
        else:
            tp, fp = 0, 0
            fn = 1 # There's at least 1 allo. pocket per PDB

        if total == 0 or not modeld["all_pockets_in_output"]:
            preds[pdb] = {
                "tp": tp,
                "fn": fn,
                "fp": fp
            }
        else:
            preds[pdb] = {
                "total": total,
                "tp": tp,
                "fn": fn,
                "fp": fp
            }

    models_preds[model] = preds

models_preds

{'model5': {'22mj': {'total': 12, 'tp': 1, 'fn': 0, 'fp': 0},
  '6s3a': {'total': 6, 'tp': 1, 'fn': 0, 'fp': 0},
  '6vvq': {'total': 13, 'tp': 1, 'fn': 0, 'fp': 0},
  '6z1m': {'total': 14, 'tp': 1, 'fn': 0, 'fp': 0},
  '7p2v': {'total': 14, 'tp': 1, 'fn': 0, 'fp': 0},
  '7xmd': {'total': 15, 'tp': 0, 'fn': 1, 'fp': 1},
  '7yg5': {'total': 66, 'tp': 0, 'fn': 1, 'fp': 1},
  '7zpe': {'total': 11, 'tp': 1, 'fn': 0, 'fp': 0},
  '8crc': {'total': 10, 'tp': 1, 'fn': 0, 'fp': 0},
  '8f4s': {'total': 10, 'tp': 1, 'fn': 0, 'fp': 0},
  '8fpi': {'total': 59, 'tp': 1, 'fn': 0, 'fp': 0},
  '8jp0': {'total': 33, 'tp': 1, 'fn': 0, 'fp': 0},
  '8qni': {'total': 17, 'tp': 1, 'fn': 0, 'fp': 0},
  '8uk6': {'total': 30, 'tp': 1, 'fn': 0, 'fp': 0},
  '8v81': {'total': 45, 'tp': 1, 'fn': 0, 'fp': 0},
  '9ebs': {'total': 13, 'tp': 1, 'fn': 0, 'fp': 0},
  '9fsj': {'total': 14, 'tp': 1, 'fn': 0, 'fp': 0},
  '9ntc': {'total': 12, 'tp': 1, 'fn': 0, 'fp': 0},
  '9oul': {'total': 17, 'tp': 1, 'fn': 0, 'fp': 0},
  '

allositepro vs allopockets
allositepro fails 6z1m, 7p2v, 7zpe, 8fpi, 9ebs, 9ntc
we both fail 7xmd, 7yg5

In [31]:
from sklearn.metrics import matthews_corrcoef, f1_score, confusion_matrix

In [32]:
models_metrics = {}

for model, preds in models_preds.items():
    df = pd.DataFrame(preds).T
    
    if models[model]["all_pockets_in_output"]:
        y_true = [1] * df["tp"].sum() + [1] * df["fn"].sum() + [0] * df["fp"].sum() + [0] * (df["total"].sum() - df[["tp", "fn", "fp"]].sum().sum())
        y_pred = [1] * df["tp"].sum() + [0] * df["fn"].sum() + [1] * df["fp"].sum() + [0] * (df["total"].sum() - df[["tp", "fn", "fp"]].sum().sum())

        models_metrics[model] = {
            "model": model,
            "tp": df["tp"].sum(),
            "fn": df["fn"].sum(),
            "fp": df["fp"].sum(),
            "mcc": matthews_corrcoef(y_true, y_pred),
            "macro-f1": f1_score(y_true, y_pred, average="macro"),
            "confmat": pd.DataFrame(confusion_matrix(y_true, y_pred))
        }
        
    else:
        models_metrics[model] = {
            "model": model,
            "tp": df["tp"].sum(),
            "fn": df["fn"].sum(),
            "fp": df["fp"].sum(),
        }

models_metrics = pd.DataFrame(models_metrics).T.sort_values(["tp", "mcc"], ascending=False)
models_metrics

,model,tp,fn,fp,mcc,macro-f1,confmat
model5,model5,18,2,2,0.895157,0.947579,0 1 0 411 2 1 2 18
allositepro,allositepro,12,8,3,NaN,NaN,NaN


In [69]:
models_metrics.infer_objects().to_csv("models_metrics_topx.csv", index=False, sep="\t", decimal=",")

# Old

In [79]:
model = "passer_ensemble"
modeld = models["passer_ensemble"]

results = modeld["results"]
results

{'7gqu': {'20': {'pred': 1,
   'prob/score': 50.99665205925703,
   'residues':    auth_asym_id auth_seq_id pdbx_PDB_ins_code
   0             A         916                 ?
   1             A         917                 ?
   4             A         920                 ?
   5             A         727                 ?
   6             A         919                 ?
   8             A         555                 ?
   9             A         552                 ?
   11            A         551                 ?
   12            A         846                 ?
   13            A         845                 ?
   15            A         849                 ?
   17            A         913                 ?
   25            A         706                 ?
   26            A         570                 ?
   27            A         726                 ?
   29            A         725                 ?
   30            A         898                 ?
   31            A         895            

In [84]:
modeld["prob_key"]

'prob/score'

In [80]:
if modeld["labelling"] is not None:
    labelled_results = label_results(results, **modeld["labelling"], prob_key=modeld["prob_key"])
# else:
#     labelled_results = pd.DataFrame((
#         {
#             "pdb": pdb,
#             "pocket": pocket,
#             **pocketd
#         }
#         for pdb, pockets in results.items()
#         for pocket, pocketd in pockets.items()
#     ))
labelled_results

,pdb,pocket,prob,pred,label,max_overlap,pocket_in_site,site_in_pocket
262,8uk6,1,71.084872,1,1,0.904762,0.904762,0.904762
306,8v81,49,42.582709,0,1,0.888889,0.666667,0.888889
241,8qni,13,48.563351,1,1,0.833333,0.510204,0.833333
0,7gqu,20,50.996652,1,1,0.758621,0.687500,0.758621
141,8aq6,8,46.686667,0,1,0.740741,0.740741,0.689655
...,...,...,...,...,...,...,...,...
133,7yg5,69,3.204734,0,0,0.000000,0.000000,0.000000
131,7yg5,58,3.321074,0,0,0.000000,0.000000,0.000000
130,7yg5,86,3.379763,0,0,0.000000,0.000000,0.000000
129,7yg5,44,3.398566,0,0,0.000000,0.000000,0.000000


In [81]:
labelled_results.sort_values("pred", ascending=False).iloc[:40]

,pdb,pocket,prob,pred,label,max_overlap,pocket_in_site,site_in_pocket
378,9dnm,20,57.407993,1,0,0.000000,0.000000,0.000000
140,8aq6,18,49.659825,1,0,0.137931,0.102564,0.137931
304,8v81,73,61.714813,1,0,0.000000,0.000000,0.000000
182,8jp0,57,51.007666,1,1,0.718750,0.534884,0.718750
158,8f4s,9,50.438614,1,1,0.730769,0.730769,0.730769
0,7gqu,20,50.996652,1,1,0.758621,0.687500,0.758621
241,8qni,13,48.563351,1,1,0.833333,0.510204,0.833333
27,7yg5,46,58.276781,1,0,0.000000,0.000000,0.000000
262,8uk6,1,71.084872,1,1,0.904762,0.904762,0.904762
354,8v81,17,4.550745,0,0,0.000000,0.000000,0.000000


In [ ]:
preds = {}
for pdb in news:
    if pdb == "9dnm": continue #######################################################################################
    pdbpreds = labelled_results.query(f"pdb == '{pdb}'")
    total = len(pdbpreds)
    if total > 0:
        # Top1 pred for our model
        if model == "model5":
            pdbpreds.loc[:, "pred"] = pdbpreds[["prob"]].apply(lambda x: (x == x.max()).astype(int)).values
        elif model == "allofusion":
            pdbpreds = pdbpreds.drop(
                pdbpreds.loc[lambda x: (x.pred == 1) & (x.label == 1)]
                .index[1:]
            )
        tp = len( pdbpreds.loc[lambda x: (x["pred"] == 1) & (x["label"] == 1)] )
        fp = len( pdbpreds.loc[lambda x: (x["pred"] == 1) & (x["label"] == 0)] )
        fn = len( pdbpreds.loc[lambda x: (x["pred"] == 0) & (x["label"] == 1)] )
        if tp + fn == 0:
            fn = 1 # There's at least 1 allo. pocket per PDB
    else:
        tp, fp = 0, 0
        fn = 1 # There's at least 1 allo. pocket per PDB

    if total == 0 or not modeld["all_pockets_in_output"]:
        preds[pdb] = {
            "tp": tp,
            "fn": fn,
            "fp": fp
        }
    else:
        preds[pdb] = {
            "total": total,
            "tp": tp,
            "fn": fn,
            "fp": fp
        }

models_preds[model] = preds

In [71]:
from sklearn.metrics import matthews_corrcoef, f1_score, confusion_matrix

In [72]:
models_metrics = {}

for model, preds in models_preds.items():
    df = pd.DataFrame(preds).T
    
    if models[model]["all_pockets_in_output"]:
        y_true = [1] * df["tp"].sum() + [1] * df["fn"].sum() + [0] * df["fp"].sum() + [0] * (df["total"].sum() - df[["tp", "fn", "fp"]].sum().sum())
        y_pred = [1] * df["tp"].sum() + [0] * df["fn"].sum() + [1] * df["fp"].sum() + [0] * (df["total"].sum() - df[["tp", "fn", "fp"]].sum().sum())

        models_metrics[model] = {
            "model": model,
            "tp": df["tp"].sum(),
            "fn": df["fn"].sum(),
            "fp": df["fp"].sum(),
            "mcc": matthews_corrcoef(y_true, y_pred),
            "macro-f1": f1_score(y_true, y_pred, average="macro"),
            "confmat": pd.DataFrame(confusion_matrix(y_true, y_pred))
        }
        
    else:
        models_metrics[model] = {
            "model": model,
            "tp": df["tp"].sum(),
            "fn": df["fn"].sum(),
            "fp": df["fp"].sum(),
        }

models_metrics = pd.DataFrame(models_metrics).T.sort_values(["tp", "mcc"], ascending=False)
models_metrics

,model,tp,fn,fp,mcc,macro-f1,confmat
model5,model5,7,1,1,0.870495,0.935248,0 1 0 221 1 1 1 7
allositepro,allositepro,7,1,1,NaN,NaN,NaN
passer_ensemble,passer_ensemble,5,3,3,0.616892,0.808446,0 1 0 367 3 1 3 5
allofusion,allofusion,5,3,56,NaN,NaN,NaN
passer_automl,passer_automl,4,4,4,0.489189,0.744595,0 1 0 366 4 1 4 4
alloses,alloses,3,5,5,0.36145,0.680725,0 1 0 364 5 1 5 3
mefallosite,mefallosite,3,5,5,0.358553,0.679276,0 1 0 299 5 1 5 3
allo,allo,3,5,5,0.346751,0.673376,0 1 0 172 5 1 5 3
deepallo,deepallo,2,6,6,0.233827,0.616914,0 1 0 365 6 1 6 2
stingallo,stingallo,2,6,1,NaN,NaN,NaN


In [65]:
models_metrics.infer_objects().to_csv("models_metrics.csv", index=False, sep="\t", decimal=",")